Get averages for times of day

In [1]:
#read in relevant packages
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyproj #for utm
#pyproj.datadir.get_data_dir()
pyproj.datadir.set_data_dir('C:\\Users\\rpkamakura\\AppData\\Local\\anaconda3\\envs\\wrf\\Library\\share\\proj') #should be whatever env you're using
#to make sure you can see the env: https://stackoverflow.com/questions/53004311/how-to-add-conda-environment-to-jupyter-lab 

from netCDF4 import Dataset #for using wrf.getvar

##dates
import datetime

#to get a list of the files in the folder
import glob 

##attempt to reformat the netcdf in one line
import xwrf

#to quickly calculate relative humidity
from wrf import getvar, interplevel, to_np, latlon_coords, uvmet
import wrf


Going to need to try to average at different time points to get an overall sense of the differences

Also want to actually store those values somewhere, with information on the land use and distance from the coast for each point

Then you can do a quick regression to see how the differences between the two time points (MHW - non) vary with land use and distance to the water, with pixel location as a random effect. 

Steps for this include:
* mask out the water
* shift the dataframe to a row for each pixel at each hour time point with temperature as the value (and/or humidity)
* add the land use and distance to water as constants
* run the regression

Resources:
* averaging: https://stackoverflow.com/questions/67868777/averaging-multiple-netcdf4-files-with-python

In [8]:
##trying this with wrf-python instead to see if it makes it at all easier
#d02_2016_names = glob.glob("../../01Data/Houston_MarineHeatWave_2016/wrfout_MarineHeatWave.d02.*")
#d02_2017_names = glob.glob("../../01Data/Houston_MarineHeatWave_2017/wrfout_MarineHeatWave.d02.*")
d02_2017_climavg_names  = glob.glob("../../01Data/Houston_MarineHeatWave_2017_w16SST/wrfout_MarineHeatWave.d02.*")

##Ideally we want to ignore the first few days as spin up (maybe get rid of 2 days worth)

#pull out the datasets
#wrfin = [Dataset(f) for f in d02_2016_names[8:] ] #gives you two days of burn-in time I think
#initial_ds = xr.open_dataset(d02_2016_names[0]).xwrf.postprocess() #for land use
#wrfin = [Dataset(f) for f in d02_2017_names[8:] ] #gives you two days of burn-in time I think
#initial_ds = xr.open_dataset(d02_2017_names[0]).xwrf.postprocess() #for land use
wrfin = [Dataset(f) for f in d02_2017_climavg_names[8:] ] #gives you two days of burn-in time I think
initial_ds = xr.open_dataset(d02_2017_climavg_names[0]).xwrf.postprocess() #for land use

T2_data = wrf.getvar(wrfin, "T2", timeidx=wrf.ALL_TIMES)
rh2_data = wrf.getvar(wrfin, "rh2", timeidx=wrf.ALL_TIMES)

#Uwind_data = wrf.getvar(wrfin, "U", timeidx=wrf.ALL_TIMES)
#Vwind_data = wrf.getvar(wrfin, "V", timeidx=wrf.ALL_TIMES)

# Get grid-relative U and V
uv = getvar(wrfin, "uvmet", timeidx=wrf.ALL_TIMES) 
uv10m = getvar(wrfin, "uvmet10", timeidx=wrf.ALL_TIMES)

#wind speed
wspd =  getvar(wrfin, "uvmet_wspd_wdir", units="kt", timeidx=wrf.ALL_TIMES)[0,:]

#pressure
p = getvar(wrfin, "pressure", timeidx=wrf.ALL_TIMES)         # shape: (Time, bottom_top, south_north, west_east)
uv_850 = interplevel(uv, p, 850)

#trying to get clouds
#these are the thresholds with height, but we don't have that, we have pressure: https://www.weather.gov/key/cloudchart
clouds = getvar(wrfin, "cloudfrac", timeidx=wrf.ALL_TIMES)  
llclouds = clouds[0,:,:,:]

#sea level pressure
slp = wrf.getvar(wrfin, "slp", timeidx=wrf.ALL_TIMES)
#have to manage the shortwave accumulation a bit differently here...

In [9]:
llclouds

<xarray.DataArray 'cloudfrac' (Time: 193, south_north: 225, west_east: 230)> Size: 40MB
array([[[0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ]],

       [[0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
...
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ]],

       [[0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ]]], dtype=float32)
Coordinates:
    XLONG         (south_north, west_east) float32 207kB -96.51 -96.5 ... -94.18
    XLAT          (south_north, west_east) float32 207kB 28.71 28.71 ... 30.69
    XTIME         (Time) float64 2kB 2.88e+03 2.94e+03 ... 1.434e+04 1.44e+04
  * Time          (Time) datetime64[ns] 2kB 2017-03-25 ... 2017-04-02
    low_mid_high  <U4 16B 'low'
Dimensions without coordinates: south_north, west_east
Attributes:
    FieldType:      104
    MemoryOrder:    XY
    description:    low, mid, high clouds
    units:          %
    stagger:        
    coordinates:    XLONG XLAT XTIME
    projection:     Mercator(stand_lon=0.0, moad_cen_lat=28.746002197265625, ...
    low_thresh:     300.0 m
    mid_thresh:     2000.0 m
    high_thresh:    6000.0 m
    _FillValue:     9.969209968386869e+36
    missing_value:  9.969209968386869e+36

Now go and extract the values of these variables across the whole grid for each hour

In [3]:
##now get the average values for each hour
DatesList = T2_data.coords['Time'].values
HourList = pd.to_datetime(DatesList).hour

#prep data for land use
init_ds = initial_ds.isel(Time=0)
landuse = init_ds.LU_INDEX

# Variables and datasets
variables = {
    'T2': T2_data,
    'rh2': rh2_data,
    'uv10': uv10m,
    'uv850': uv_850,
    'llc': llclouds,
    'wspd': wspd,
    'slp': slp,
}
# Prepare storage for results
results = []

# Loop over hours
for h in range(24):
    
    hr_pos = np.where(HourList == h)[0]
    
    if len(hr_pos) == 0:
        continue

    var_means = {}

    # Compute mean across time for each variable
    for var_name, data in variables.items():
        selected = data.isel(Time=hr_pos)
        
        if var_name == 'sw' and h == 0:
            hr_pos = hr_pos[1:]  # skip first time point for SW
            selected = data.sel(Time=hr_pos)
            
        var_means[var_name] = selected.mean(dim='Time', skipna=True)

    # Grid info
    lat_vals = var_means['T2'].coords['XLAT'].values
    lon_vals = var_means['T2'].coords['XLONG'].values

    ny, nx = landuse.shape  # typically (south_north, west_east)
    uv10 = var_means["uv10"]
    uv850 = var_means["uv850"]
    wspd = var_means["wspd"]

    for i in range(ny):       # south_north
        for j in range(nx):   # west_east
            lat = lat_vals[i, j]
            lon = lon_vals[i, j]

            #for the winds
            row = {
                'hour': h,
                'lat': float(lat),
                'lon': float(lon),
                'landuse': int(landuse.values[i, j])
            }
            
            row["u10"] = uv10.values[0, i, j].item()
            row["v10"] = uv10.values[1, i, j].item()
            row["u850"] = uv850.values[0, i, j].item()
            row["v850"] = uv850.values[1, i, j].item()
            row["wspd"] = wspd.values[0, i, j].item()
            
            for var_name, mean_arr in var_means.items():

                if var_name =="uv10" or var_name =="uv850" or var_name =="wspd":
                    continue
                    
                row[var_name] = mean_arr.values[i, j].item()
            results.append(row)


# Convert to DataFrame
df = pd.DataFrame(results)


# Optional: save to CSV
#df.to_csv("hourly_grid_averages.csv", index=False)

In [4]:
#df.to_csv('../../03ProcessedData/LargeFiles/HourlyMeans_2016_070125.csv', index=False)
#df.to_csv('../../03ProcessedData/LargeFiles/HourlyMeans_2017_070125.csv', index=False)
df.to_csv('../../03ProcessedData/LargeFiles/HourlyMeans_2017_w16SST_070125.csv', index=False)